# PokerMon: Deep CFR Training (TPU)

Train a Deep CFR strategy network for heads-up No-Limit Texas Hold'em on a **Colab TPU**.

**Runtime**: Go to `Runtime > Change runtime type` and select **TPU v2**.

**Data**: No external data needed. Deep CFR generates its own training data by traversing the game tree.

**Output**: A `strategy_net` checkpoint saved to Google Drive that the PokerMon web frontend loads.

## 1. Mount Google Drive

Checkpoints are saved to Drive so training survives runtime disconnects.

In [2]:
from google.colab import drive
drive.mount('/content/drive')

import os
import sys
DRIVE_DIR = '/content/drive/MyDrive/PokerMon'
os.makedirs(f'{DRIVE_DIR}/checkpoints', exist_ok=True)
os.makedirs(f'{DRIVE_DIR}/runs', exist_ok=True)
print(f'Checkpoints: {DRIVE_DIR}/checkpoints/')
print(f'TensorBoard logs: {DRIVE_DIR}/runs/')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Checkpoints: /content/drive/MyDrive/PokerMon/checkpoints/
TensorBoard logs: /content/drive/MyDrive/PokerMon/runs/


# Install PyTorch XLA for TPU support
!pip install torch_xla -q

# Clone PokerMon and install as editable package
import os, sys
if not os.path.exists('/content/PokerMon'):
    !git clone https://github.com/KenWuqianghao/PokerMon.git /content/PokerMon
else:
    !cd /content/PokerMon && git pull

!pip install -e /content/PokerMon -q

# Ensure the module is importable in the running kernel
if '/content/PokerMon' not in sys.path:
    sys.path.insert(0, '/content/PokerMon')

import pokermon
print(f'PokerMon installed: {pokermon.__file__}')

In [3]:
# Install PyTorch XLA for TPU support
!pip install torch_xla -q

# Clone PokerMon and install as editable package
import os
if not os.path.exists('/content/PokerMon'):
    !git clone https://github.com/kenwuqianghao/PokerMon.git /content/PokerMon
else:
    !cd /content/PokerMon && git pull

!pip install -e /content/PokerMon -q

# Ensure the module is importable in the running kernel
if '/content/PokerMon' not in sys.path:
    sys.path.insert(0, '/content/PokerMon')

import pokermon
print(f'PokerMon installed: {pokermon.__file__}')

Already up to date.
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for pokermon (pyproject.toml) ... done
PokerMon installed: /content/PokerMon/pokermon/__init__.py


## 3. Verify TPU

In [4]:
import torch
import torch_xla
import torch_xla.core.xla_model as xm

device = xm.xla_device()
print(f'PyTorch: {torch.__version__}')
print(f'torch_xla: {torch_xla.__version__}')
print(f'XLA device: {device}')

# Sanity check
t = torch.randn(2, 2, device=device)
print(f'Tensor on TPU: {t.device}')
print('TPU is ready!')

/tmp/ipython-input-4964/3121990614.py:5: DeprecationWarning: Use torch_xla.device instead
  device = xm.xla_device()


PyTorch: 2.9.0+cpu
torch_xla: 2.9.0
XLA device: xla:0
Tensor on TPU: xla:0
TPU is ready!


## 4. Configure Training

Adjust `num_iterations` and `traversals_per_iter` to control training duration and model strength.

In [5]:
from pokermon.train.config import TrainConfig

DRIVE_DIR = '/content/drive/MyDrive/PokerMon'

config = TrainConfig(
    # Game: heads-up NLHE (2 players for web frontend)
    game='nlhe_hu',
    num_players=2,
    small_blind=50,
    big_blind=100,
    starting_stack=10_000,

    # Deep CFR iterations
    num_iterations=100,
    traversals_per_iter=5_000,

    # Network architecture
    hidden_dim=256,
    num_layers=4,
    num_actions=7,

    # Training
    advantage_sgd_steps=2000,
    strategy_sgd_steps=2000,
    strategy_train_every=5,
    lr=1e-3,
    batch_size=2048,
    buffer_capacity=1_000_000,
    weight_exponent=1.5,

    # Pruning
    prune_after=50,
    prune_threshold=-3e8,

    # Persistent storage on Google Drive
    checkpoint_dir=f'{DRIVE_DIR}/checkpoints/nlhe_hu',
    checkpoint_every=25,
    log_dir=f'{DRIVE_DIR}/runs/nlhe_hu',

    # TPU via PyTorch XLA
    device='xla',
    seed=42,
)

print(f'Device: {config.resolve_device()}')
print(f'Iterations: {config.num_iterations}')
print(f'Traversals/iter: {config.traversals_per_iter}')
print(f'Network: {config.num_layers} layers x {config.hidden_dim} hidden')
print(f'Buffer capacity: {config.buffer_capacity:,}')
print(f'Checkpoints: {config.checkpoint_dir}')

Device: xla
Iterations: 100
Traversals/iter: 5000
Network: 4 layers x 256 hidden
Buffer capacity: 1,000,000
Checkpoints: /content/drive/MyDrive/PokerMon/checkpoints/nlhe_hu


## 5. Train

Checkpoints save to Drive every 25 iterations. If the runtime disconnects, resume from the last checkpoint (Section 8).

The first few iterations may be slower as XLA compiles computation graphs.

In [ ]:
from pokermon.train.trainer import Trainer

trainer = Trainer(config)
trainer.train()
print('Training complete!')

Deep CFR NLHE:   0%|          | 0/100 [00:00<?, ?it/s]

## 6. Evaluate

Sample actions from the strategy net to verify it learned non-uniform strategies.

In [ ]:
import numpy as np
from pokermon.cfr.infoset import encode_infoset_flat
from pokermon.game.action import Action
from pokermon.game.engine import new_hand, get_legal_actions_mask, get_legal_actions

strategy_net = trainer.strategy_net
strategy_net.eval()
device = trainer.device

print('Strategy samples from trained model:\n')
for i in range(10):
    state = new_hand(num_players=2, small_blind=50, big_blind=100, stacks=[10000, 10000])
    player = state.current_player

    features = encode_infoset_flat(state, player)
    mask = get_legal_actions_mask(state)
    legal = get_legal_actions(state)

    with torch.no_grad():
        x = torch.tensor(features, dtype=torch.float32, device=device).unsqueeze(0)
        m = torch.tensor(mask, dtype=torch.bool, device=device).unsqueeze(0)
        probs = strategy_net(x, m).squeeze(0).cpu().numpy()

    print(f'Hand {i+1} (player {player}):')
    for a in legal:
        print(f'  {a.name:12s} {probs[a]*100:5.1f}%')
    print()

## 7. Export Checkpoint for Web Frontend

Save the final model to Drive in the format the web server expects. Models are moved to CPU before saving so the checkpoint loads on any device.

In [ ]:
import os
from pathlib import Path

DRIVE_DIR = '/content/drive/MyDrive/PokerMon'
export_path = Path(DRIVE_DIR) / 'smoke_test.pt'

# Move to CPU before saving so checkpoint is device-agnostic
cpu_strategy = trainer.strategy_net.cpu()
cpu_advantage = [n.cpu() for n in trainer.advantage_nets]

checkpoint = {
    'strategy_net': cpu_strategy.state_dict(),
    'advantage_nets': [n.state_dict() for n in cpu_advantage],
    'iteration': config.num_iterations,
    'config': {
        'hidden_dim': config.hidden_dim,
        'num_layers': config.num_layers,
        'num_actions': config.num_actions,
        'num_players': config.num_players,
        'num_iterations': config.num_iterations,
        'traversals_per_iter': config.traversals_per_iter,
    },
}
torch.save(checkpoint, export_path)

size_mb = os.path.getsize(export_path) / 1e6
print(f'Saved: {export_path} ({size_mb:.1f} MB)')
print()
print('To use in the web frontend:')
print('  1. Download smoke_test.pt from Google Drive')
print('  2. Copy to: checkpoints/nlhe6/smoke_test.pt')
print('  3. Restart: uvicorn server.app:app --reload')

In [ ]:
# Download directly to your computer
from google.colab import files
files.download(str(export_path))

## 8. Resume Training (if runtime disconnected)

Run this section to load from the latest Drive checkpoint and continue training.

In [ ]:
import glob
from pokermon.train.trainer import Trainer
from pokermon.train.config import TrainConfig
from pokermon.train.checkpoint import load_checkpoint

DRIVE_DIR = '/content/drive/MyDrive/PokerMon'

ckpts = sorted(glob.glob(f'{DRIVE_DIR}/checkpoints/nlhe_hu/checkpoint_*.pt'))
if not ckpts:
    print('No checkpoints found on Drive. Run Section 5 first.')
else:
    latest = ckpts[-1]
    print(f'Found {len(ckpts)} checkpoint(s). Latest: {latest}')

    resume_config = TrainConfig(
        game='nlhe_hu',
        num_players=2,
        small_blind=50,
        big_blind=100,
        starting_stack=10_000,
        num_iterations=200,
        traversals_per_iter=5_000,
        hidden_dim=256,
        num_layers=4,
        num_actions=7,
        advantage_sgd_steps=2000,
        strategy_sgd_steps=2000,
        strategy_train_every=5,
        lr=1e-3,
        batch_size=2048,
        buffer_capacity=1_000_000,
        checkpoint_dir=f'{DRIVE_DIR}/checkpoints/nlhe_hu',
        checkpoint_every=25,
        log_dir=f'{DRIVE_DIR}/runs/nlhe_hu_resumed',
        device='xla',
        seed=42,
    )

    trainer = Trainer(resume_config)
    info = load_checkpoint(latest, trainer.advantage_nets, trainer.strategy_net)
    print(f'Resumed from iteration {info["iteration"]}')
    trainer.train()
    print('Resume training complete!')

## 9. TensorBoard (Optional)

In [ ]:
DRIVE_DIR = '/content/drive/MyDrive/PokerMon'
%load_ext tensorboard
%tensorboard --logdir {DRIVE_DIR}/runs